<div style="font-family: 'Helvetica Neue', -apple-system, Arial, sans-serif;">

### Playground to Test: Do Signals have Predictive Power?
___

</div>

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))

from modules.signals.fetch import get_sp500_universe, fetch_price_data, START_DATE, END_DATE
from modules.signals.momentum import compute_momentum
from modules.signals.mean_reversion import compute_mean_reversion
from modules.signals.testing import evaluate_signal

import pandas as pd

In [ ]:
from modules.signals.fetch import save_price_data

universe = get_sp500_universe()
print(f"{len(universe)} tickers across {universe['Sector'].nunique()} sectors")
close, volume = fetch_price_data(universe["Symbol"].tolist(), start=START_DATE, end=END_DATE)

failed_tickers = close.columns[close.isna().all()].tolist()
if failed_tickers:
    close = close.drop(columns=failed_tickers)
    volume = volume.drop(columns=failed_tickers)

save_price_data(universe, close, volume)
print(f"Close shape: {close.shape}")

# once the data's been saved...

# from modules.signals.fetch import load_price_data

# universe, close, volume = load_price_data()
# print(f"Close shape: {close.shape}")
# close.tail()

Fetching batch 1 / 13 (40 tickers)...


[*********************100%***********************]  40 of 40 completed


Fetching batch 2 / 13 (40 tickers)...


[*********************100%***********************]  40 of 40 completed


Fetching batch 3 / 13 (40 tickers)...


...

Fetching batch 12 / 13 (40 tickers)...


[*********************100%***********************]  40 of 40 completed


Fetching batch 13 / 13 (23 tickers)...


[*********************100%***********************]  23 of 23 completed


Saved universe, close, and volume to /Users/paulgrajzl/Documents/Quant-Projects/systematic-portfolio-engine-equities-1/data/
Close shape: (1907, 503)


In [5]:
momentum_signal = compute_momentum(close)
mean_reversion_signal = compute_mean_reversion(close)

print(f"Momentum shape: {momentum_signal.shape}, non-null: {momentum_signal.notna().sum().sum()}")
print(f"Mean Reversion shape: {mean_reversion_signal.shape}, non-null: {mean_reversion_signal.notna().sum().sum()}")

Momentum shape: (1907, 503), non-null: 907326
Mean Reversion shape: (1907, 503), non-null: 936240


In [6]:
HORIZON = 20  # arbitrary starting point, just to confirm the pipeline runs

mom_ic_series, mom_summary = evaluate_signal(momentum_signal, close, horizon=HORIZON)
print(pd.Series(mom_summary).round(4))

Mean IC                -0.0147
IC Std                  0.1954
Information Ratio      -0.0754
% Positive IC           0.5126
N Observations       1824.0000
dtype: float64


In [7]:
mr_ic_series, mr_summary = evaluate_signal(mean_reversion_signal, close, horizon=HORIZON)
print(pd.Series(mr_summary).round(4))

Mean IC                 0.0077
IC Std                  0.1766
Information Ratio       0.0437
% Positive IC           0.5138
N Observations       1882.0000
dtype: float64


In [8]:
from modules.standardization.zscore import zscore_signal, winsorize_signal, standardize_signal

In [9]:
momentum_standardized = standardize_signal(momentum_signal)
mean_reversion_standardized = standardize_signal(mean_reversion_signal)

print("Momentum standardized:")
print(momentum_standardized.iloc[-1].describe())

Momentum standardized:
count    501.000000
mean      -0.022428
std        0.906576
min       -2.509160
25%       -0.625165
50%       -0.068416
75%        0.442767
max        3.000000
Name: 2026-08-04 00:00:00, dtype: float64


In [10]:
# Pick a random date and confirm the standardization worked as intended
check_date = momentum_standardized.dropna(how="all").index[-1]

print(f"Checking date: {check_date}")
print(f"Mean: {momentum_standardized.loc[check_date].mean():.4f} (should be close to 0)")
print(f"Std: {momentum_standardized.loc[check_date].std():.4f} (should be close to 1)")
print(f"Min: {momentum_standardized.loc[check_date].min():.4f}, Max: {momentum_standardized.loc[check_date].max():.4f} (should be within +/-3 after winsorizing)")

Checking date: 2026-08-04 00:00:00
Mean: -0.0224 (should be close to 0)
Std: 0.9066 (should be close to 1)
Min: -2.5092, Max: 3.0000 (should be within +/-3 after winsorizing)


In [11]:
mom_standardized_ic, mom_standardized_summary = evaluate_signal(momentum_standardized, close, horizon=HORIZON)
print("Momentum (standardized) IC summary:")
print(pd.Series(mom_standardized_summary).round(4))

print("\nMomentum (raw) IC summary, for comparison:")
print(pd.Series(mom_summary).round(4))

Momentum (standardized) IC summary:
Mean IC                -0.0147
IC Std                  0.1954
Information Ratio      -0.0754
% Positive IC           0.5126
N Observations       1824.0000
dtype: float64

Momentum (raw) IC summary, for comparison:
Mean IC                -0.0147
IC Std                  0.1954
Information Ratio      -0.0754
% Positive IC           0.5126
N Observations       1824.0000
dtype: float64


In [12]:
from modules.construction.simple_construction import quantile_weights, score_proportional_weights

In [13]:
momentum_quantile_weights = quantile_weights(momentum_standardized, long_pct=0.2, short_pct=0.2, gross_exposure=1.0)

check_date = momentum_quantile_weights[(momentum_quantile_weights != 0).any(axis=1)].index[-1]
print(f"Checking date: {check_date}")

nonzero_weights = momentum_quantile_weights.loc[check_date]
nonzero_weights = nonzero_weights[nonzero_weights != 0]

print(f"Number of positions: {len(nonzero_weights)}")
print(f"Sum of long weights: {nonzero_weights[nonzero_weights > 0].sum():.4f} (should be ~0.5)")
print(f"Sum of short weights: {nonzero_weights[nonzero_weights < 0].sum():.4f} (should be ~-0.5)")
print(f"Net exposure: {nonzero_weights.sum():.4f} (should be ~0, market-neutral by construction)")
print(f"Gross exposure: {nonzero_weights.abs().sum():.4f} (should be ~1.0)")

Checking date: 2026-08-04 00:00:00
Number of positions: 200
Sum of long weights: 0.5000 (should be ~0.5)
Sum of short weights: -0.5000 (should be ~-0.5)
Net exposure: 0.0000 (should be ~0, market-neutral by construction)
Gross exposure: 1.0000 (should be ~1.0)


In [14]:
momentum_proportional_weights = score_proportional_weights(momentum_standardized, gross_exposure=1.0)

nonzero_prop_weights = momentum_proportional_weights.loc[check_date]
nonzero_prop_weights = nonzero_prop_weights[nonzero_prop_weights != 0]

print(f"Number of positions: {len(nonzero_prop_weights)}")
print(f"Sum of long weights: {nonzero_prop_weights[nonzero_prop_weights > 0].sum():.4f} (should be ~0.5)")
print(f"Sum of short weights: {nonzero_prop_weights[nonzero_prop_weights < 0].sum():.4f} (should be ~-0.5)")
print(f"Net exposure: {nonzero_prop_weights.sum():.4f} (should be ~0)")
print(f"Gross exposure: {nonzero_prop_weights.abs().sum():.4f} (should be ~1.0)")

Number of positions: 501
Sum of long weights: 0.5000 (should be ~0.5)
Sum of short weights: -0.5000 (should be ~-0.5)
Net exposure: -0.0000 (should be ~0)
Gross exposure: 1.0000 (should be ~1.0)


In [15]:
mr_quantile_weights = quantile_weights(mean_reversion_standardized, long_pct=0.2, short_pct=0.2, gross_exposure=1.0)

nonzero_mr_weights = mr_quantile_weights.loc[check_date]
nonzero_mr_weights = nonzero_mr_weights[nonzero_mr_weights != 0]

print(f"Number of positions: {len(nonzero_mr_weights)}")
print(f"Net exposure: {nonzero_mr_weights.sum():.4f}")
print(f"Gross exposure: {nonzero_mr_weights.abs().sum():.4f}")

Number of positions: 200
Net exposure: -0.0000
Gross exposure: 1.0000


<div style="font-family: 'Helvetica Neue', -apple-system, Arial, sans-serif;">

### Dollar-Neutral vs. Beta-Neutral

*Dollar-neutral:* long dollar exposure equals short dollar exposure (net exposure ≈ 0). This is what construction module gives us automatically, as a side effect of the 50/50 long-short split.

*Beta-neutral:* the long side's *aggregate market sensitivity* equals the short side's, so the portfolio doesn't gain or lose purely from the overall market moving up or down. Dollar-neutrality does NOT guarantee this. If the longs happen to be higher-beta stocks than the shorts, the portfolio still has an unintended market-direction bet, even at zero net dollar exposure.

At this current moment, we're dollar-neutral only. Beta-neutrality (and sector-neutrality) hasn't been enforced yet — that's the job of the risk module next.
___

</div>

In [16]:
# Most recent date with actual positions
latest_date = momentum_quantile_weights[(momentum_quantile_weights != 0).any(axis=1)].index[-1]
latest_positions = momentum_quantile_weights.loc[latest_date]
latest_positions = latest_positions[latest_positions != 0].sort_values(ascending=False)

print(f"Positions as of: {latest_date}\n")

print("Top 5 Long:")
print(latest_positions.head(5))

print("\nTop 5 Short:")
print(latest_positions.tail(5))

Positions as of: 2026-08-04 00:00:00

Top 5 Long:
Ticker
A       0.005
HPQ     0.005
MET     0.005
LRCX    0.005
LH      0.005
Name: 2026-08-04 00:00:00, dtype: float64

Top 5 Short:
Ticker
KR     -0.005
CME    -0.005
CMS    -0.005
ISRG   -0.005
ZTS    -0.005
Name: 2026-08-04 00:00:00, dtype: float64
